To run this code:

**Requirements:** Python 3.11 and above.

The first cell can be run to install all required packages from `requirements.txt`. You also need a `.env` file in the same folder as this notebook containing the following credentials:

* Open Electricity API
* Neon PostgreSQL info
* HiveMQ Cloud MQTT Broker info

> ⚠️ **Note:** This notebook requires a local environment (Jupyter Notebook / VS Code / Cursor). Google Colab is not supported due to local MQTT and interactive widgets (ipyleaflet, ipywidgets).

> If you're using VScode, press 'Enable Downloads' for the dash part: `jupyter-leaflet`

## 0 Setup

### 0-1 Environment

In [1]:
%pip install -r requirements.txt

import os # if user has anaconda Arrow11 library, this is to avoid conflict
os.environ["ARROW_HOME"] = ""
os.environ["ARROW_WITH_ACERO"] = "ON"

%pip install pyarrow --only-binary=pyarrow

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


### 0-2 Import packages and key

In [2]:
import logging, os, json, io, base64, time, threading
import psycopg2, psycopg2.extras
import pandas as pd
import polars as pl
import paho.mqtt.client as mqtt
import matplotlib.pyplot as plt
import ipywidgets as w
import ipyleaflet as L
import certifi, ssl, aiohttp
import openelectricity.client as _oe_client
from openelectricity import OEClient
from openelectricity.types import DataMetric, MarketMetric
from openelectricity.client import APIError
from datetime import datetime, timedelta
from itertools import groupby
from dotenv import load_dotenv
from pathlib import Path
from IPython.display import display, clear_output, HTML as IPHTML
import nest_asyncio

# CA certification for Open Electricity API
class _CertifiClientSession(aiohttp.ClientSession):
    def __init__(self, *args, connector=None, **kwargs):
        if connector is None:
            connector = aiohttp.TCPConnector(ssl=ssl.create_default_context(cafile=certifi.where()))
        super().__init__(*args, connector=connector, **kwargs)

_oe_client.ClientSession = _CertifiClientSession

# Get the confidential info (password, API key...)
load_dotenv(Path(".env"))

# for OEClient: can run asyncio loop (avoids Jupyter "loop already running")
nest_asyncio.apply()  

# don't print the log``
logging.getLogger("openelectricity").setLevel(logging.WARNING)


/var/folders/rq/dq6k05k570l01_l2vrdmp3440000gn/T/ipykernel_18788/4196429581.py:22: DeprecationWarning: Inheritance class _CertifiClientSession from ClientSession is discouraged
  class _CertifiClientSession(aiohttp.ClientSession):


## 1 Data Acquisition

### 1-1 Support Functions (Def)

In [3]:
start_at = datetime(2026, 5, 1)
end_at   = datetime(2026, 5, 8)

In [4]:
def response_to_df(response, unit_to_facility=None) -> pl.DataFrame:
    """OE API response into long-format Polars"""
    records: dict[tuple, dict] = {}

    for series in response.data:
        for result in series.results:
            unit_code      = result.columns.unit_code
            network_region = result.columns.network_region

            if unit_code and unit_to_facility is not None:
                identifier = unit_to_facility.get(unit_code, unit_code)
            elif unit_code:
                identifier = unit_code
            elif network_region:
                identifier = network_region
            else:
                parts      = result.name.split("_", 1)
                identifier = parts[1] if len(parts) == 2 else result.name

            for p in result.data:
                key = (p.timestamp, identifier)  # 一列 = 一個時間點 × 一個設施/區域
                row = records.setdefault(key, {"interval": p.timestamp, "code": identifier})
                prev = row.get(series.metric)
                if prev is not None and p.value is not None:
                    row[series.metric] = prev + p.value
                elif series.metric not in row:
                    row[series.metric] = p.value

    return pl.DataFrame(list(records.values())) if records else pl.DataFrame()


### 1-2 Get Facility (Enrichment) Info

In [5]:
facility_codes    = []
unit_to_facility  = {}
facility_meta_rows = []

# Get facility static info
with OEClient() as client:
    facilities = client.get_facilities(network_id=["NEM"])

# Get detailed info and Tidy up
for fac in facilities.data:
    facility_codes.append(fac.code)

    units = getattr(fac, "units", []) or []
    fuels = [getattr(u, "fueltech_id", None) for u in units if getattr(u, "fueltech_id", None)]
    fuels = [f.value if hasattr(f, "value") else str(f) for f in fuels]
    primary_fuel = max(set(fuels), key=fuels.count) if fuels else None

    loc = getattr(fac, "location", None)
    facility_meta_rows.append({
        "code":           fac.code,
        "facility_name":  getattr(fac, "name", None),
        "primary_fuel":   primary_fuel,
        "network_region": getattr(fac, "network_region", None),
        "lat":            loc.lat if loc else None,
        "lng":            loc.lng if loc else None,
    })

    for unit in units:
        unit_to_facility[unit.code] = fac.code

facility_meta = pl.DataFrame(facility_meta_rows)
print(f"Total NEM facilities: {len(facility_codes)}")
print(f"Unit→facility mappings: {len(unit_to_facility)}")
print(facility_meta.head(5))


Total NEM facilities: 542
Unit→facility mappings: 958
shape: (5, 6)
┌──────────┬───────────────────────┬───────────────┬────────────────┬────────────┬────────────┐
│ code     ┆ facility_name         ┆ primary_fuel  ┆ network_region ┆ lat        ┆ lng        │
│ ---      ┆ ---                   ┆ ---           ┆ ---            ┆ ---        ┆ ---        │
│ str      ┆ str                   ┆ str           ┆ str            ┆ f64        ┆ f64        │
╞══════════╪═══════════════════════╪═══════════════╪════════════════╪════════════╪════════════╡
│ ADP      ┆ Adelaide Desalination ┆ solar_utility ┆ SA1            ┆ -35.100751 ┆ 138.483357 │
│ ALDGASF  ┆ Aldoga                ┆ solar_utility ┆ QLD1           ┆ -23.839544 ┆ 151.0849   │
│ AMCORGR  ┆ Amcor Glass           ┆ distillate    ┆ SA1            ┆ -34.882663 ┆ 138.577975 │
│ ANGASTON ┆ Angaston              ┆ distillate    ┆ SA1            ┆ -34.503389 ┆ 139.02458  │
│ APS      ┆ Anglesea              ┆ coal_brown    ┆ VIC1           

### 1-3 Get Power and Emission Per Facility

In [6]:
BATCH_SIZE = 30  # API calls upper limit
batches    = [facility_codes[i:i+BATCH_SIZE] for i in range(0, len(facility_codes), BATCH_SIZE)]

all_dfs = []
print(f"Running {len(batches)} batches, {len(facility_codes)} facilities...")

for i, batch in enumerate(batches):
    # for each batch (30 facilities): call OE API
    with OEClient() as client:
        resp = client.get_facility_data(
            network_code="NEM",
            facility_code=batch, # batch: list of 30 facility codes
            metrics=[DataMetric.POWER, DataMetric.EMISSIONS],
            interval="5m",
            date_start=start_at,
            date_end=end_at, 
        )
    df_batch = response_to_df(resp, unit_to_facility=unit_to_facility)
    all_dfs.append(df_batch)
    if (i + 1) % 5 == 0 or i == len(batches) - 1:
        print(f"  batch {i+1}/{len(batches)} done — {df_batch.shape[0]} rows")

df_facility = pl.concat(all_dfs).join(facility_meta, on="code", how="left")
print(f"\nFinal shape: {df_facility.shape}")
df_facility.head()


Running 19 batches, 542 facilities...
  batch 5/19 done — 40320 rows
  batch 10/19 done — 46368 rows
  batch 15/19 done — 36288 rows
  batch 19/19 done — 2016 rows

Final shape: (716454, 9)


interval,code,power,emissions,facility_name,primary_fuel,network_region,lat,lng
"datetime[μs, UTC]",str,f64,f64,str,str,str,f64,f64
2026-04-30 14:00:00 UTC,"""ADP""",0.0,0.0,"""Adelaide Desalination""","""solar_utility""","""SA1""",-35.100751,138.483357
2026-04-30 14:05:00 UTC,"""ADP""",0.0,0.0,"""Adelaide Desalination""","""solar_utility""","""SA1""",-35.100751,138.483357
2026-04-30 14:10:00 UTC,"""ADP""",0.212,0.0,"""Adelaide Desalination""","""solar_utility""","""SA1""",-35.100751,138.483357
2026-04-30 14:15:00 UTC,"""ADP""",0.0,0.0,"""Adelaide Desalination""","""solar_utility""","""SA1""",-35.100751,138.483357
2026-04-30 14:20:00 UTC,"""ADP""",0.0,0.0,"""Adelaide Desalination""","""solar_utility""","""SA1""",-35.100751,138.483357


### 1-4 Get Market Price and Demand

In [7]:
with OEClient() as client:
    market_response = client.get_market(
        network_code="NEM",
        metrics=[MarketMetric.PRICE, MarketMetric.DEMAND],
        interval="5m",
        date_start=start_at,
        date_end=end_at,
        primary_grouping="network_region",
    )

df_market = response_to_df(market_response).rename({"code": "network_region"})
print(df_market.shape)
df_market.head()

(12096, 4)


interval,network_region,price,demand
"datetime[μs, UTC]",str,f64,f64
2026-04-30 14:00:00 UTC,"""NSW1""",57.51,7332.5
2026-04-30 14:05:00 UTC,"""NSW1""",57.51,7323.78
2026-04-30 14:10:00 UTC,"""NSW1""",65.89,7441.25
2026-04-30 14:15:00 UTC,"""NSW1""",65.22,7417.92
2026-04-30 14:20:00 UTC,"""NSW1""",57.43,7387.76


### 1-5 Get Facility Info from Assi1

In [8]:
# Get data from Assi1 (Cloud DB)
NEON = {
    "host":            os.environ["NEON_HOST"],
    "port":            int(os.environ.get("NEON_PORT", "5432")),
    "user":            os.environ.get("NEON_USER", "neondb_owner"),
    "password":        os.environ["NEON_PASSWORD"],
    "sslmode":         "require",
    "connect_timeout": 20,
    "dbname":          os.environ.get("NEON_DBNAME", "neondb"),
}

conn = psycopg2.connect(**NEON)
fac_geo = pd.read_sql(
    "SELECT facility_name, latitude, longitude, state, postcode FROM dim_facility",
    conn
)
conn.close()
print(f"Loaded {len(fac_geo)} facilities from Neon DB.")
fac_geo.head()

# for later 2-2 df combine
geo = (
    pl.from_pandas(fac_geo)
    .select(
        pl.col("facility_name").str.strip_chars(),
        pl.col("latitude").alias("lat_a1"),
        pl.col("longitude").alias("lng_a1"),
    )
    .unique(subset=["facility_name"])
)

/var/folders/rq/dq6k05k570l01_l2vrdmp3440000gn/T/ipykernel_18788/4146924198.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fac_geo = pd.read_sql(


Loaded 860 facilities from Neon DB.


## 2. Data Cleaning and Integration

### 2-1 Quick View Before Cleaning

In [9]:
pl.Config.set_tbl_cols(9)
pl.Config.set_tbl_width_chars(500)
print(df_facility.head(10))
print(df_market.head(10))

shape: (10, 9)
┌─────────────────────────┬──────┬───────┬───────────┬───────────────────────┬───────────────┬────────────────┬────────────┬────────────┐
│ interval                ┆ code ┆ power ┆ emissions ┆ facility_name         ┆ primary_fuel  ┆ network_region ┆ lat        ┆ lng        │
│ ---                     ┆ ---  ┆ ---   ┆ ---       ┆ ---                   ┆ ---           ┆ ---            ┆ ---        ┆ ---        │
│ datetime[μs, UTC]       ┆ str  ┆ f64   ┆ f64       ┆ str                   ┆ str           ┆ str            ┆ f64        ┆ f64        │
╞═════════════════════════╪══════╪═══════╪═══════════╪═══════════════════════╪═══════════════╪════════════════╪════════════╪════════════╡
│ 2026-04-30 14:00:00 UTC ┆ ADP  ┆ 0.0   ┆ 0.0       ┆ Adelaide Desalination ┆ solar_utility ┆ SA1            ┆ -35.100751 ┆ 138.483357 │
│ 2026-04-30 14:05:00 UTC ┆ ADP  ┆ 0.0   ┆ 0.0       ┆ Adelaide Desalination ┆ solar_utility ┆ SA1            ┆ -35.100751 ┆ 138.483357 │
│ 2026-04-30 14:10:

### 2-2 Cleaning: Inconsistent and Irrelevant

In [10]:
# drop duplicate data
df_facility = df_facility.unique(subset=["interval", "code"])
df_market   = df_market.unique(subset=["interval", "network_region"])
# normalise
df_facility = df_facility.with_columns(
    pl.col("emissions").clip(lower_bound=0),
    pl.col("primary_fuel").str.strip_chars().str.to_lowercase(),
)
df_market = df_market.filter(pl.col("network_region") != "WEM")

# Drop facilities that never produced power
active_codes = (
    df_facility.group_by("code")
    .agg((pl.col("power") != 0).any().alias("active"))
    .filter(pl.col("active"))["code"]
)
n_dropped   = df_facility["code"].n_unique() - len(active_codes)
df_facility = (
    df_facility
    .filter(pl.col("code").is_in(active_codes))
    .filter(pl.col("facility_name").is_not_null())
    .filter(pl.col("primary_fuel").is_not_null())
)
# Enrich with Assignment 1's lat/lng, Drop the one without neither from both
df_facility = (
    df_facility
    .with_columns(pl.col("facility_name").str.strip_chars())
    .join(geo, on="facility_name", how="left")
    .with_columns(
        pl.col("lat").fill_null(pl.col("lat_a1")),
        pl.col("lng").fill_null(pl.col("lng_a1")),
    )
    .drop("lat_a1", "lng_a1")
)
df_facility = df_facility.filter(
    pl.col("lat").is_not_null() & pl.col("lng").is_not_null()
)

print(f"Dropped {n_dropped} constant-zero facilities → {df_facility['code'].n_unique()} active")
print(f"After cleaning → facility: {df_facility.shape}, market: {df_market.shape}")


Dropped 30 constant-zero facilities → 326 active
After cleaning → facility: (657206, 9), market: (10080, 4)


/var/folders/rq/dq6k05k570l01_l2vrdmp3440000gn/T/ipykernel_18788/1400194578.py:20: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .filter(pl.col("code").is_in(active_codes))


### 2-3 Handling: Missing data and format

In [11]:
# fill missing data
df_facility = df_facility.with_columns(
    pl.col("power").fill_null(0),
    pl.col("emissions").fill_null(0),
)

# fill missing with forward/backward fill
df_market = (
    df_market.sort(["network_region", "interval"])
    .with_columns(
        pl.col("price").forward_fill().over("network_region"),
        pl.col("demand").forward_fill().over("network_region"),
    )
    .with_columns(
        pl.col("price").backward_fill().over("network_region"),
        pl.col("demand").backward_fill().over("network_region"),
    )
)

# Strip timezone and standardise column name
df_facility = df_facility.with_columns(pl.col("interval").dt.replace_time_zone(None)).rename({"interval": "time_interval_utc"})
df_market   = df_market.with_columns(pl.col("interval").dt.replace_time_zone(None)).rename({"interval": "time_interval_utc"})

print(f"Clean df_facility: {df_facility.shape}")
print(df_facility.head(3))
print(f"\nClean df_market: {df_market.shape}")
print(df_market.head(3))

Clean df_facility: (657206, 9)
shape: (3, 9)
┌─────────────────────┬──────────┬───────┬───────────┬───────────────┬───────────────┬────────────────┬────────────┬────────────┐
│ time_interval_utc   ┆ code     ┆ power ┆ emissions ┆ facility_name ┆ primary_fuel  ┆ network_region ┆ lat        ┆ lng        │
│ ---                 ┆ ---      ┆ ---   ┆ ---       ┆ ---           ┆ ---           ┆ ---            ┆ ---        ┆ ---        │
│ datetime[μs]        ┆ str      ┆ f64   ┆ f64       ┆ str           ┆ str           ┆ str            ┆ f64        ┆ f64        │
╞═════════════════════╪══════════╪═══════╪═══════════╪═══════════════╪═══════════════╪════════════════╪════════════╪════════════╡
│ 2026-05-03 02:00:00 ┆ BUTLERSG ┆ 6.6   ┆ 0.0       ┆ Butlers Gorge ┆ hydro         ┆ TAS1           ┆ -42.266898 ┆ 146.261823 │
│ 2026-05-05 21:05:00 ┆ SMCSF    ┆ 0.0   ┆ 0.0       ┆ Sun Metals    ┆ solar_utility ┆ QLD1           ┆ -19.332335 ┆ 146.875929 │
│ 2026-05-01 01:55:00 ┆ LKBONNY3 ┆ 0.0   ┆ 0.

### 2-4 Data Store

In [12]:
# create data folder
os.makedirs("data", exist_ok=True)

df_publish = (
    df_facility
    .join(df_market, on=["time_interval_utc", "network_region"], how="left")
    .rename({"price": "market_price", "demand": "market_demand"})
)
df_publish.write_csv("data/facility_market.csv", float_precision=5)
print(f"Saved {df_publish.shape[0]:,} rows → data/facility_market.csv")

Saved 657,206 rows → data/facility_market.csv


## 3. Data Publish via MQTT

### 3-1 Publish Dataset Schema

In [13]:
conn = psycopg2.connect(**NEON)
cur  = conn.cursor()

cur.execute("""
    CREATE TABLE IF NOT EXISTS fact_market (
        time_interval_utc TIMESTAMP       NOT NULL,
        facility_code     TEXT            NOT NULL,
        power_mw          DOUBLE PRECISION,
        emissions_tco2    DOUBLE PRECISION,
        facility_name     TEXT,
        primary_fuel      TEXT,
        network_region    TEXT,
        lat               DOUBLE PRECISION,
        lng               DOUBLE PRECISION,
        market_price_mwh  DOUBLE PRECISION,
        market_demand_mw  DOUBLE PRECISION,
        PRIMARY KEY (time_interval_utc, facility_code)
    );
""")
conn.commit()
print("Schema created: fact_market")
cur.close()
conn.close()

print(f"df_publish shape : {df_publish.shape}")
print(f"Columns          : {df_publish.columns}")
print(f"Time range       : {df_publish['time_interval_utc'].min()} → {df_publish['time_interval_utc'].max()}")
df_publish.head(3)

Schema created: fact_market
df_publish shape : (657206, 11)
Columns          : ['time_interval_utc', 'code', 'power', 'emissions', 'facility_name', 'primary_fuel', 'network_region', 'lat', 'lng', 'market_price', 'market_demand']
Time range       : 2026-04-30 14:00:00 → 2026-05-07 13:55:00


time_interval_utc,code,power,emissions,…,lat,lng,market_price,market_demand
datetime[μs],str,f64,f64,…,f64,f64,f64,f64
2026-05-03 02:00:00,"""BUTLERSG""",6.6,0.0,…,-42.266898,146.261823,95.0,1037.11
2026-05-05 21:05:00,"""SMCSF""",0.0,0.0,…,-19.332335,146.875929,164.82,6668.67
2026-05-01 01:55:00,"""LKBONNY3""",0.0,0.0,…,-37.760504,140.405541,-6.9,949.7


### 3-2 MQTT Publish

In [14]:
# MQTT config
MQTT_BROKER     = os.environ.get("MQTT_BROKER", "broker.hivemq.com")
MQTT_PORT       = int(os.environ.get("MQTT_PORT", "1883"))
MQTT_TOPIC_BASE = "INFO5995A2/ccha0131"
MAX_INFLIGHT    = 50
ACK_TIMEOUT     = 120


def _publish_qos1(client, topic, payload):
    #follow QoS1: if failed -> retry later
    while True:
        info = client.publish(topic, payload, qos=1, retain=False)
        if info.rc == mqtt.MQTTErrorCode.MQTT_ERR_SUCCESS:
            return info
        time.sleep(0.01)


_last_published = {}   # (code, time_interval_str) -> value sig; used when delta=True


def _publish_loop(wait=0.0, delta=False):
    global _last_published

    # Build row list
    all_rows = df_publish.sort("time_interval_utc").to_dicts()  # sort by time -> publish by time
    # for section 6 : only publish the changed rows
    if delta:
        new_snapshot = {}
        sorted_rows  = []
        for row in all_rows:
            ts  = str(row["time_interval_utc"])
            key = (row["code"], ts)
            sig = (row["power"], row["emissions"],
                   row.get("market_price"), row.get("market_demand"))
            new_snapshot[key] = sig
            if _last_published.get(key) != sig:   # new or changed
                sorted_rows.append(row)
        print(f"  Delta: {len(sorted_rows):,} / {len(all_rows):,} records changed")
    else:
        sorted_rows = all_rows

    # MQTT publish 
    client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
    client.max_inflight_messages_set(MAX_INFLIGHT)  # max info : 50 waiting for ACK
    client.max_queued_messages_set(0)               # no queue
    client.username_pw_set(os.environ["MQTT_USERNAME"], os.environ["MQTT_PASSWORD"])
    client.tls_set(ca_certs=certifi.where())
    client.connect(MQTT_BROKER, MQTT_PORT, keepalive=60)
    client.loop_start()

    topic   = f"{MQTT_TOPIC_BASE}/stream"
    pending = []
    total   = 0

    # make each time a group -> for loop
    for i, (ts, batch) in enumerate(groupby(sorted_rows, key=lambda r: str(r["time_interval_utc"]))):
        for row in batch:
            payload = json.dumps({
                "time_interval_utc": ts,
                "facility_code":     row["code"],
                "facility_name":     row["facility_name"],
                "primary_fuel":      row["primary_fuel"],
                "network_region":    row["network_region"],
                "lat":               row.get("lat"),
                "lng":               row.get("lng"),
                "power_mw":          row["power"],
                "emissions_tco2":    row["emissions"],
                "market_price_mwh":  row.get("market_price"),
                "market_demand_mw":  row.get("market_demand"),
            }, default=str)
            pending.append(_publish_qos1(client, topic, payload))
            total += 1
            if wait:
                time.sleep(wait)
            if len(pending) >= MAX_INFLIGHT:  # when >= 50 waiting ACK: pop oldest one
                pending.pop(0).wait_for_publish(timeout=ACK_TIMEOUT)
        if (i + 1) % 50 == 0:
            print(f"  [{i+1}] {ts} — {total:,} msgs sent")

    # before stop: wait if there are pendings
    for info in pending:
        info.wait_for_publish(timeout=ACK_TIMEOUT)
    # stop (this round) client
    client.loop_stop() 
    client.disconnect()

    if delta:
        _last_published = new_snapshot   # save snapshot for next round's comparison
    print(f"Done. {total:,} messages → {MQTT_BROKER}:{MQTT_PORT}")

# background running
threading.Thread(target=lambda: _publish_loop(wait=0.1), daemon=True).start()
print(f"Publishing {df_publish.shape[0]:,} records to {MQTT_TOPIC_BASE}/stream")


Publishing 657,206 records to INFO5995A2/ccha0131/stream


## 4. Map-based Dashboard

### 4-1 MQTT Subscriber

In [15]:
# Shared state: MQTT subscriber write, HTML use, dashboard poll read
facility_records_by_code = {}
map_markers_by_code      = {}
selected_facility_code   = [None]
current_map_popup        = [None]

mqtt_data_lock         = threading.Lock()
mqtt_facility_records  = {}
facility_history       = {}
mqtt_message_count     = [0]

# default: stop all background polling first
_dashboard_poll_stop = threading.Event()
_dashboard_poll_stop.set()
_poll_last_count = [0]
DASHBOARD_POLL_S = 2.0
_facility_stop = threading.Event()
_facility_stop.set()
_energy_stop   = threading.Event()
_energy_stop.set()
_market_stop   = threading.Event()
_market_stop.set()


In [ ]:
MQTT_BROKER    = os.environ.get("MQTT_BROKER", "broker.hivemq.com")
MQTT_PORT      = int(os.environ.get("MQTT_PORT", "1883"))
MQTT_TOPIC_SUB = "INFO5995A2/ccha0131/stream"

# subscriber_ready signals to the publisher (3-2) 
subscriber_ready = threading.Event()

# stop existing subscriber, if any -> avoid duplicate subscription
try:
    sub_client.loop_stop()
    sub_client.disconnect()
except Exception:
    pass


def on_connect(client, userdata, flags, rc, properties=None):
    # Paho subscriber topic after connect
    if rc == 0: #success
        client.subscribe(MQTT_TOPIC_SUB, qos=1)
        print(f"Subscribed to {MQTT_TOPIC_SUB}")
        subscriber_ready.set()
    else:
        print(f"MQTT connection failed (rc={rc})")


def on_message(client, userdata, msg):
    # write after receive each JSON data
    try:
        d = json.loads(msg.payload)
    except json.JSONDecodeError as e:
        print(f"MQTT JSON error: {e}")
        return
    code = d.get("facility_code")
    if not code:
        return

    time_interval = d.get("time_interval_utc") or ""
    record = {
        "time_interval_utc": time_interval,
        "facility_code":     code,
        "facility_name":     d.get("facility_name"),
        "primary_fuel":      d.get("primary_fuel"),
        "network_region":    d.get("network_region"),
        "lat":               d.get("lat"),
        "lng":               d.get("lng"),
        "power_mw":          d.get("power_mw"),
        "emissions_tco2":    d.get("emissions_tco2"),
        "market_price_mwh":  d.get("market_price_mwh"),
        "market_demand_mw":  d.get("market_demand_mw"),
        "received_at":       datetime.now().isoformat(timespec="seconds"),
    }
    with mqtt_data_lock:  # 與 5-5 poll、5-4 讀取共用鎖
        if time_interval:
            facility_history[(code, time_interval)] = record
        mqtt_facility_records[code] = record
        mqtt_message_count[0] += 1
        count = mqtt_message_count[0]
    if count == 1:
        print(f"  first MQTT message received ({code})")
    elif count % 2000 == 0:
        print(f"  received {count} messages, {len(mqtt_facility_records)} facilities")


sub_client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
sub_client.on_connect = on_connect
sub_client.on_message = on_message
sub_client.username_pw_set(os.environ["MQTT_USERNAME"], os.environ["MQTT_PASSWORD"])
sub_client.tls_set(ca_certs=certifi.where())
sub_client.connect(MQTT_BROKER, MQTT_PORT, keepalive=60)
sub_client.loop_start()
print("MQTT subscriber running in background.")
threading.Timer(3.0, lambda: print(f"  MQTT buffer: {mqtt_message_count[0]} msgs, {len(mqtt_facility_records)} facilities")).start()


MQTT subscriber running in background.


Subscribed to INFO5995A2/ccha0131/stream
  first MQTT message received (MURRAY)
  received 2000 messages, 326 facilities
  received 4000 messages, 326 facilities


### 4-2 Dash HTML/CSS Setting

#### 4-2-1 Static CSS

In [17]:
# Overall Color
FUEL_COLOR = {
    "solar_utility":       "#FFD700",
    "wind":                "#00BFFF",
    "gas_ccgt":            "#FF6347",
    "gas_ocgt":            "#FF4500",
    "gas_steam":           "#FF7F50",
    "coal_black":          "#222222",
    "coal_brown":          "#8B4513",
    "hydro":               "#4169E1",
    "battery_charging":    "#9370DB",
    "battery_discharging": "#7B68EE",
    "bioenergy_biogas":    "#228B22",
    "bioenergy_biomass":   "#2E8B57",
    "distillate":          "#DAA520",
}
DEFAULT_FUEL_COLOR = "#888888"

# Main CSS
MapDash_CSS = """<style>
  .widget-button { transition: filter 0.08s !important; }
  .widget-button:hover { box-shadow: none !important; }
  .widget-button:focus { outline: none !important; box-shadow: none !important; }
  .widget-button:active { filter: brightness(0.88) !important; box-shadow: none !important; }

  /* L1 header card */
  .dash-header-card {
    display: flex !important; align-items: center !important;
    justify-content: space-between !important;
    padding: 6px 10px !important; background: #fff !important;
    width: 100% !important; box-sizing: border-box !important;
    margin-bottom: 4px !important;
  }
  .dash-L1-left { display: flex !important; align-items: center !important; gap: 8px !important; }
  .dash-L1-right { display: flex !important; align-items: center !important; gap: 6px !important; }

  /* L1 left toggle pills */
  .dash-toggle-pill .widget-button {
    border-radius: 30px !important;
    border: none !important;
    padding: 4px 16px !important;
    color: #3e4958 !important;
    font-size: 13px !important;
    white-space: nowrap !important;
    box-shadow: none !important;
  }
  .dash-toggle-pill .widget-button:hover { filter: brightness(0.90) !important; }

  /* L2 Map */
  .dashboard-map {
    width: 100% !important;
    max-width: 100% !important;
    height: 400px !important;
    max-height: 400px !important;
    overflow: hidden !important;
    box-sizing: border-box !important;
  }
  @media (max-width: 1000px) {
    .dashboard-map { height: 320px !important; max-height: 320px !important; }
  }
  @media (max-width: 700px) {
    .dashboard-map { height: 240px !important; max-height: 240px !important; }
  }
  .dashboard-map .leaflet-container {
    width: 100% !important; height: 100% !important; min-height: 0 !important;
  }

  /* L3 table */
  .dash-L3-table { flex: 0 0 auto !important; }
  .dash-L3-body .widget-html-content {
    overflow: visible !important;
    height: auto !important;
    max-height: none !important;
  }
  .dash-L3-body .ts-table-scroll {
    display: block !important;
    height: 260px !important;
    max-height: 260px !important;
    overflow-y: auto !important;
    overflow-x: hidden !important;
    background: #fff !important;
    color-scheme: light !important;
    scrollbar-width: thin !important;
    scrollbar-color: #c0c0c0 #f5f5f5 !important;
    box-sizing: border-box !important;
  }
  .dash-L3-body .ts-table-scroll::-webkit-scrollbar { width: 12px; }
  .dash-L3-body .ts-table-scroll::-webkit-scrollbar-track { background: #f5f5f5; }
  .dash-L3-body .ts-table-scroll::-webkit-scrollbar-thumb {
    background: #c0c0c0; border-radius: 6px;
  }

  /* Shrink-to-fit: avoid horizontal overflow */
  .dash-box-root, .dash-box-root .p-Widget, .dash-box-root .jupyter-widget {
    min-width: 0 !important;
    max-width: 100% !important;
    box-sizing: border-box !important;
  }
  .dash-L1-left, .dash-L1-right { min-width: 0 !important; flex-shrink: 1 !important; }
  .jp-OutputArea-child:has(.dash-box-root) .jp-OutputArea-output,
  .jp-OutputArea-child:has(.dash-box-root) .output_subarea {
    overflow-x: hidden !important;
    scrollbar-width: none !important;
  }

</style>"""

# L2 Map Static Setting
REGION_CENTER = {
    "NSW1": {"center": [-33.0, 147.0], "zoom": 5},
    "VIC1": {"center": [-37.0, 144.5], "zoom": 7},
    "QLD1": {"center": [-23.0, 144.0], "zoom": 5},
    "SA1":  {"center": [-32.5, 137.0], "zoom": 6},
    "TAS1": {"center": [-42.0, 146.5], "zoom": 7},
}
MAP_DEFAULT_CENTER = [-27, 145]
MAP_DEFAULT_ZOOM   = 4
MAP_L2_HEIGHT      = "400px" 

# L2 Map Popup Card CSS 
POPUP_CONTENT = (
    "font-family:'Roboto',Arial,sans-serif;"
    "padding:2px 12px 5px 4px;min-width:145px;max-width:195px;"
    "text-align:center;line-height:1.3;"
)
POPUP_NAME = (
    "font-size:20px;font-weight:700;color:#000;"
    "margin:0 0 4px;line-height:1.2;"
)
POPUP_META       = "font-size:12px;color:#333;margin-bottom:3px;"
POPUP_FUEL_DOT   = "font-size:12px;"
POPUP_TS         = "font-size:10px;color:#9c9c9c;margin-bottom:6px;"
POPUP_DATABOX = (
    "background:rgba(217,217,217,0.5);border-radius:6px;"
    "padding:5px 9px;font-size:11px;color:#111;line-height:1.35;text-align:center;"
)
POPUP_MIN_WIDTH = 155  # Leaflet L.Popup min_width 

# L2 map marker style
MARKER_OPACITY          = 0.75
MARKER_WEIGHT           = 1
MARKER_SEL_OPACITY      = 1.0
MARKER_SEL_WEIGHT       = 3
MARKER_SEL_BORDER_COLOR = "#222"
MARKER_SEL_RADIUS_BONUS = 4

# L3 table static setting
COLS      = ["Facility Name", "Fuel Type", "Power (MW)", "Emissions (tCO2)", "Region", "Price/MWh", "Demand (MW)"]
COL_W     = [190, 118, 88, 108, 70, 112, 89]
COL_ALIGN = ["left", "center", "right", "right", "center", "right", "right"]

HEADER_BTN_INACTIVE = "#ebebeb"
HEADER_BTN_ACTIVE   = "#c0c0c0"

table_sort_col = ["Power (MW)"]
table_sort_asc = [False]

# L3 Table CSS
TABLE_TD  = "padding:4px 8px;font-size:12px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap"
TABLE_ROW = "border-bottom:1px solid #eee"
# L3 table body scroll
TABLE_SCROLL_BAR_W = 12
TABLE_SCROLL_STYLE = (
    "display:block;height:260px;max-height:260px;overflow-y:auto;overflow-x:hidden;"
    "background:#fff;color-scheme:light;scrollbar-width:thin;scrollbar-color:#c0c0c0 #f5f5f5;"
)
TABLE_SCROLL_CSS = """<style>
.ts-table-scroll::-webkit-scrollbar{width:12px}
.ts-table-scroll::-webkit-scrollbar-track{background:#f5f5f5}
.ts-table-scroll::-webkit-scrollbar-thumb{background:#c0c0c0;border-radius:6px}
</style>"""

# L4 Date Label CSS
DATA_UPTO_STYLE = "text-align:right;color:#555;font-size:12px;padding:4px 2px 0"


#### 4-2-2 UI Layout Widgets

In [18]:
# ── L1 Header — widgets ───────────────────────────────────────────────────────
_toggle_state = ["Power (MW)"]
_btn_power = w.Button(description="Power (MW)",      layout=w.Layout(height="30px"))
_btn_emis  = w.Button(description="Emission (tCO2)", layout=w.Layout(height="30px"))
for _b in [_btn_power, _btn_emis]:
    _b.add_class("dash-toggle-pill")

L1_left_toggle_widget = w.HBox(
    [_btn_power, _btn_emis],
    layout=w.Layout(align_items="center"),
)

L1_region_dropdown = w.Dropdown(
    options=["All"], value="All", description="",
    layout=w.Layout(width="120px"),
)
L1_fuel_dropdown = w.Dropdown(
    options=["All"], value="All", description="",
    layout=w.Layout(width="168px"),
)

L1_left_label = w.HTML(
    "<span style='font-size:14px;font-weight:700;color:#3e4958;"
    "white-space:nowrap'>Circle Size by:</span>",
    layout=w.Layout(align_self="center"),
)
L1_left_button = w.HBox(
    [L1_left_label, L1_left_toggle_widget],
    layout=w.Layout(align_items="center", gap="8px"),
)
L1_left_button.add_class("dash-L1-left")

L1_region_label = w.HTML(
    "<span style='font-size:13px;color:#3e4958;white-space:nowrap'>Region:</span>",
    layout=w.Layout(align_self="center"),
)
L1_fuel_label = w.HTML(
    "<span style='font-size:13px;color:#3e4958;white-space:nowrap'>Fuel:</span>",
    layout=w.Layout(align_self="center"),
)
L1_right_dropdown = w.HBox(
    [L1_region_label, L1_region_dropdown, L1_fuel_label, L1_fuel_dropdown],
    layout=w.Layout(align_items="center", gap="6px", margin="0 0 0 auto"),
)
L1_right_dropdown.add_class("dash-L1-right")

L1_header = w.HBox(
    [L1_left_button, L1_right_dropdown],
    layout=w.Layout(width="100%", min_width="0", align_items="center"),
)
L1_header.add_class("dash-header-card")


# ── L2 Map — widgets (instance created in Map Dashboard init_L2_map) ───────────

L2_map = None
L2_marker_layer = None


# ── L3 Table — widgets ────────────────────────────────────────────────────────

L3_header_buttons = [
    w.Button(
        description=f"{col} ↕",
        layout=w.Layout(flex=f"{wp} 1 0%", min_width="0", height="32px", padding="0 2px"),
    )
    for col, wp in zip(COLS, COL_W)
]
for btn in L3_header_buttons:
    btn.style.button_color = HEADER_BTN_INACTIVE
    btn.style.font_weight  = "600"
    btn.style.text_align   = "center"
    btn.add_class("table-header-btn")

L3_scrollbar_pad = w.Box(
    layout=w.Layout(
        flex=f"0 0 {TABLE_SCROLL_BAR_W}px",
        width=f"{TABLE_SCROLL_BAR_W}px",
        background_color="#f5f5f5",
    ),
)
L3_header_row = w.HBox(
    L3_header_buttons + [L3_scrollbar_pad],
    layout=w.Layout(width="100%", min_width="0", border="1px solid #ccc"),
)
L3_table_html = w.HTML()
L3_body_box = w.VBox(
    [L3_table_html],
    layout=w.Layout(
        width="100%", min_width="0", border="1px solid #ccc",
        border_top="none", border_radius="0 0 4px 4px",
        min_height="262px", overflow="visible",
    ),
)
L3_body_box.add_class("dash-L3-body")
L3_table = w.VBox(
    [L3_header_row, L3_body_box],
    layout=w.Layout(width="100%", min_width="0", margin="0", flex="0 0 auto"),
)
L3_table.add_class("dash-L3-table")


# ── L4 Footer — widget ────────────────────────────────────────────────────────

L4_date_label = w.HTML(layout=w.Layout(width="100%", height="28px", margin="0"))


# ── Dashboard shell (_dash_box) ───────────────────────────────────────────────

# Depends on L1_header, L3_table, L4_date_label above. 
L2_map_slot = w.HTML(value="", layout=w.Layout(width="100%", height="0px", margin="0"))
_dash_box = w.VBox(
    [L1_header, L2_map_slot, L3_table, L4_date_label],
    layout=w.Layout(width="100%", max_width="100%", min_width="0"),
)
_dash_box.add_class("dash-box-root")
None  # prevent Jupyter from auto-displaying _dash_box


#### 4-2-3 Button Def Function

In [19]:
# ── L1 toggle (defs + wire buttons; callbacks registered in 5-4) ────────────────

def _sync_toggle():
    for _b in [_btn_power, _btn_emis]:
        _active = _b.description == _toggle_state[0]
        _b.style.button_color = "rgba(219,219,254,0.9)" if _active else "rgba(219,219,254,0.35)"
        _b.style.font_weight = "700" if _active else "500"

def _on_toggle(b):
    old = _toggle_state[0]
    _toggle_state[0] = b.description
    _sync_toggle()
    if old != _toggle_state[0]:
        dismiss_map_popup()
        update_visible_map_markers()

def register_l1_toggle_buttons():
    _btn_power.on_click(_on_toggle)
    _btn_emis.on_click(_on_toggle)
    _sync_toggle()


# ── L1 dropdown ─────────────────────────────────────────
def dropdown_update_map_and_table(change=None):
    """Region/fuel filter changed: pan map (if region) + refresh markers + table."""
    dismiss_map_popup()
    if L2_map is None:
        return
    region = L1_region_dropdown.value
    if region != "All" and region in REGION_CENTER:
        rc = REGION_CENTER[region]
        L2_map.center = rc["center"]
        L2_map.zoom   = rc["zoom"]
    elif region == "All":
        L2_map.center = MAP_DEFAULT_CENTER
        L2_map.zoom   = MAP_DEFAULT_ZOOM
    update_visible_map_markers()
    render_facility_table()

def refresh_filter_dropdowns():
    """Rebuild Region/Fuel dropdown options from facility_records_by_code."""
    regions = ["All"] + sorted({
        d.get("network_region")
        for d in facility_records_by_code.values()
        if d.get("network_region")
    })
    fuels = ["All"] + sorted({
        d.get("primary_fuel")
        for d in facility_records_by_code.values()
        if d.get("primary_fuel")
    })
    for widget, opts in [(L1_region_dropdown, regions), (L1_fuel_dropdown, fuels)]:
        if list(widget.options) == opts:
            continue
        prev = widget.value
        widget.options = opts
        if prev not in opts:
            widget.value = "All"


# ── L2 map ─────────────────────────────────────────

def init_L2_map():
    """build ipyleaflet map"""
    global L2_map, L2_marker_layer
    L2_map = L.Map(
        center=MAP_DEFAULT_CENTER,
        zoom=MAP_DEFAULT_ZOOM,
        scroll_wheel_zoom=True,
        layout=w.Layout(
            width="100%", min_width="0", height=MAP_L2_HEIGHT, flex="0 0 auto",
        ),
    )
    L2_map.add_class("dashboard-map")
    L2_marker_layer = L.LayerGroup(layers=[])
    L2_map.add(L2_marker_layer)


# ── L2 circle marker showing ─────────────────────────────────────────

def update_visible_map_markers():
    """ update visible markers accoreding L1 : only when parm change"""
    if L2_marker_layer is None:
        return
    mode     = _toggle_state[0]
    region_f = L1_region_dropdown.value
    fuel_f   = L1_fuel_dropdown.value
    visible  = []
    for code, mk in map_markers_by_code.items():
        d = facility_records_by_code.get(code, {})
        r = marker_radius(d, mode)
        if mk.radius != r:  # change only when parm change
            mk.radius = r
        if (region_f == "All" or d.get("network_region") == region_f) and \
           (fuel_f   == "All" or d.get("primary_fuel")   == fuel_f):
            visible.append(mk)
    new_layers = tuple(visible)
    if new_layers != L2_marker_layer.layers:  # change only when parm change
        L2_marker_layer.layers = new_layers

def newData_create_markers(codes: list):
    """Create CircleMarkers for new facility codes (popup opened on click, not here)."""
    mode = _toggle_state[0]
    for code in codes:
        d        = facility_records_by_code.get(code, {})
        lat, lng = d.get("lat"), d.get("lng")
        if lat is None or lng is None:
            continue
        color = fuel_color(d.get("primary_fuel", ""))
        mk = L.CircleMarker(
            location=[lat, lng],
            radius=marker_radius(d, mode),
            color=color,
            fill_color=color,
            fill=True,
            fill_opacity=MARKER_OPACITY,
            weight=MARKER_WEIGHT,
        )
        mk.on_click(lambda c=code, **kw: click_circle(c))
        map_markers_by_code[code] = mk

def marker_radius(d: dict, mode: str) -> int:
    v = d.get("power_mw" if mode.startswith("Power") else "emissions_tco2") or 0
    return int(max(5, min(22, 5 + v / 80)))

def fuel_color(primary_fuel: str) -> str:
    return FUEL_COLOR.get(primary_fuel, DEFAULT_FUEL_COLOR)


# ── L2 popup card ─────────────────────────────────────────

def click_circle(code: str):
    """Map click: highlight marker + Leaflet popup"""
    if L2_map is None:
        return
    mode = _toggle_state[0]
    prev = selected_facility_code[0]

    if prev and prev != code and prev in map_markers_by_code:
        mk_prev    = map_markers_by_code[prev]
        prev_d     = facility_records_by_code.get(prev, {})
        prev_color = fuel_color(prev_d.get("primary_fuel", ""))
        mk_prev.fill_opacity = MARKER_OPACITY
        mk_prev.weight       = MARKER_WEIGHT
        mk_prev.color        = prev_color
        mk_prev.radius       = marker_radius(prev_d, mode)

    _close_map_popup()
    selected_facility_code[0] = code

    if code not in map_markers_by_code or code not in facility_records_by_code:
        return

    mk = map_markers_by_code[code]
    d  = facility_records_by_code[code]

    mk.fill_opacity = MARKER_SEL_OPACITY
    mk.weight       = MARKER_SEL_WEIGHT
    mk.color        = MARKER_SEL_BORDER_COLOR
    mk.radius       = marker_radius(d, mode) + MARKER_SEL_RADIUS_BONUS

    lat, lng = d.get("lat"), d.get("lng")
    if lat is not None and lng is not None:
        popup = L.Popup(
            location=[lat, lng],
            child=w.HTML(
                value=build_facility_popup_html(d, code),
                layout=w.Layout(margin="0", padding="0"),
            ),
            close_button=True,
            auto_close=False,
            close_on_escape_key=True,
            min_width=POPUP_MIN_WIDTH,
        )
        L2_map.add(popup)
        current_map_popup[0] = popup


def _close_map_popup():
    if current_map_popup[0] is not None:
        if L2_map is not None:
            try:
                L2_map.remove(current_map_popup[0])
            except Exception:
                pass
        current_map_popup[0] = None


def dismiss_map_popup():
    """when L1 change, close popup and reset selected marker"""
    _close_map_popup()
    code = selected_facility_code[0]
    selected_facility_code[0] = None
    if not code or L2_map is None or code not in map_markers_by_code:
        return
    mode = _toggle_state[0]
    mk = map_markers_by_code[code]
    d = facility_records_by_code.get(code, {})
    color = fuel_color(d.get("primary_fuel", ""))
    mk.fill_opacity = MARKER_OPACITY
    mk.weight       = MARKER_WEIGHT
    mk.color        = color
    mk.radius       = marker_radius(d, mode)


def build_facility_popup_html(d: dict, code: str = None) -> str:
    code   = code or d.get("facility_code", "")
    name   = d.get("facility_name", code)
    fuel   = (d.get("primary_fuel") or "").replace("_", " ").title() or "\u2014"
    region = (d.get("network_region") or "").strip().rstrip("1")
    power, emi = d.get("power_mw"), d.get("emissions_tco2")
    price, demand = d.get("market_price_mwh"), d.get("market_demand_mw")
    ts   = str(d.get("time_interval_utc", ""))[:16]
    hex_ = fuel_color(d.get("primary_fuel", ""))

    def fmt(v, fmt_str, fallback="\u2014"):
        return fmt_str.format(v) if v is not None else fallback

    return (
        f'<div style="{POPUP_CONTENT}">'
        f'<div style="{POPUP_NAME}">{name}</div>'
        f'<div style="{POPUP_META}">'
        f'{region}&nbsp;&nbsp;<span style="color:{hex_};{POPUP_FUEL_DOT}">&#x25cf;</span>&nbsp;'
        f'<span style="font-weight:600;color:#111">{fuel}</span></div>'
        f'<div style="{POPUP_TS}">Latest {ts}</div>'
        f'<div style="{POPUP_DATABOX}">'
        f'<div>&#x26a1; <b>{fmt(power, "{:.1f} MW")}</b>&nbsp;&nbsp;&nbsp;'
        f'&#x1f3ed; <b>{fmt(emi, "{:.3f} tCO2")}</b></div>'
        f'<div>$ <b>{fmt(price, "${:.2f}/MWh")}</b>&nbsp;&nbsp;&nbsp;'
        f'&#x1f4ca; <b>{fmt(demand, "{:.0f} MW")}</b></div>'
        '</div></div>'
    )


# ── L3 table sorting ─────────────────────────────────────────

def render_facility_table():
    region_f, fuel_f   = L1_region_dropdown.value, L1_fuel_dropdown.value
    sort_col, sort_asc = table_sort_col[0], table_sort_asc[0]
    total_w  = sum(COL_W)
    col_tags = (
        "<colgroup>"
        + "".join(f"<col style='width:{ww/total_w*100:.2f}%'>" for ww in COL_W)
        + "</colgroup>"
    )

    rows = []
    for code in map_markers_by_code:
        d = facility_records_by_code.get(code, {})
        if region_f != "All" and d.get("network_region") != region_f:
            continue
        if fuel_f != "All" and d.get("primary_fuel") != fuel_f:
            continue
        fuel_raw = d.get("primary_fuel") or ""
        pri, dem = d.get("market_price_mwh"), d.get("market_demand_mw")
        rows.append({
            "Facility Name":     d.get("facility_name", ""),
            "Fuel Type":         fuel_raw.replace("_", " ").title() or "\u2014",
            "Power (MW)":        d.get("power_mw") or 0.0,
            "Emissions (tCO2)":  d.get("emissions_tco2") or 0.0,
            "Region":            d.get("network_region", "\u2014"),
            "Price/MWh":         pri,
            "Demand (MW)":       dem,
            "_fuel_color":       fuel_color(fuel_raw),
            "_pri_raw":          pri,
            "_dem_raw":          dem,
        })

    def sort_key(r):
        v = r[sort_col]
        return (1, 0) if v is None else (0, v) if isinstance(v, (int, float)) else (0, str(v).lower())

    rows.sort(key=sort_key, reverse=not sort_asc)

    body = []
    for r in rows:
        pri_s = f"${r['_pri_raw']:.2f}" if r["_pri_raw"] is not None else "\u2014"
        dem_s = f"{r['_dem_raw']:.0f}"  if r["_dem_raw"] is not None else "\u2014"
        vals = {
            "Facility Name":     r["Facility Name"],
            "Fuel Type":         r["Fuel Type"],
            "Power (MW)":        f"{r['Power (MW)']:.1f}",
            "Emissions (tCO2)":  f"{r['Emissions (tCO2)']:.3f}",
            "Region":            r["Region"],
            "Price/MWh":         pri_s,
            "Demand (MW)":       dem_s,
        }
        tds = []
        for col, align in zip(COLS, COL_ALIGN):
            style = f"{TABLE_TD};text-align:{align}"
            if col == "Fuel Type":
                style += f";color:{r['_fuel_color']};font-weight:600"
            tds.append(f"<td style='{style}'>{vals[col]}</td>")
        body.append(f"<tr style='{TABLE_ROW}'>{''.join(tds)}</tr>")

    empty = "<tr><td colspan='7' style='padding:8px;color:#888'>No facilities</td></tr>"
    L3_table_html.value = (
        TABLE_SCROLL_CSS
        + f"<div class='ts-table-scroll' style='{TABLE_SCROLL_STYLE}'>"
        + f"<table style='border-collapse:collapse;width:100%;table-layout:fixed'>"
        + f"{col_tags}<tbody>{''.join(body) if body else empty}</tbody></table></div>"
    )


def button_sort_column(col: str):
    if table_sort_col[0] == col:
        table_sort_asc[0] = not table_sort_asc[0]
    else:
        table_sort_col[0] = col
        table_sort_asc[0] = True
    refresh_L3_header_buttons()
    render_facility_table()


def refresh_L3_header_buttons():
    col, asc = table_sort_col[0], table_sort_asc[0]
    for btn, c in zip(L3_header_buttons, COLS):
        active = c == col
        btn.description = c + (" \u2191" if active and asc else " \u2193" if active else " \u2195")
        btn.style.button_color = HEADER_BTN_ACTIVE if active else HEADER_BTN_INACTIVE


# ── L4 realtime timestamp update ────────────────────────────────────────────────────────────

def update_L4_date_label():
    ts_candidates = [
        str(d["time_interval_utc"])
        for d in list(mqtt_facility_records.values()) + list(facility_records_by_code.values())
        if d.get("time_interval_utc")
    ]
    ts        = max(ts_candidates) if ts_candidates else None
    refreshed = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    ts_part   = f"Data up to: <b>{ts}</b>" if ts else "data up to \u2014"
    L4_date_label.value = (
        f"<div style='{DATA_UPTO_STYLE}'>"
        f"{ts_part} &nbsp;\u00b7&nbsp; last refresh at {refreshed}</div>"
    )


# ── Cross-layer: background polling (new data) ──────────

def newData_update_dashboard():
    """MQTT when new data: snapshot + update L1 ~ L4"""
    with mqtt_data_lock:
        snapshot = dict(mqtt_facility_records)
    new_codes = sorted(set(snapshot.keys()) - set(facility_records_by_code.keys()))
    facility_records_by_code.update(snapshot)
    if new_codes:
        newData_create_markers(new_codes)
    refresh_filter_dropdowns()
    update_visible_map_markers()
    render_facility_table()
    update_L4_date_label()

def poll_refresh():
    """check mqtt_message_count reguarly；update date only when new data coming"""
    if _dashboard_poll_stop.is_set():
        return
    with mqtt_data_lock:
        current = mqtt_message_count[0]
    # check if new data coming
    if current != _poll_last_count[0]: 
        _poll_last_count[0] = current
        newData_update_dashboard()
    # else: only update L4 refresh data label
    else:
        update_L4_date_label()
    threading.Timer(DASHBOARD_POLL_S, poll_refresh).start()



### 4-3 MQTT Receive Check

In [ ]:
with mqtt_data_lock: #prevent race condition
    _records = dict(mqtt_facility_records)
    _history = dict(facility_history)

_hist_ts = sorted({v["time_interval_utc"] for v in _history.values() if v.get("time_interval_utc")})

print(f"{'─'*48}")
print(f"  Total MQTT messages     : {mqtt_message_count[0]:,}")
print(f"  Unique facilities       : {len(_records)}")
print(f"  History entries         : {len(_history):,}")
print(f"  Unique timestamps       : {len(_hist_ts)}")
if _hist_ts:
    print(f"  Timestamp min           : {_hist_ts[0]}")
    print(f"  Timestamp max           : {_hist_ts[-1]}")
else:
    print(f"  Timestamps              : (none — run 3-2 first)")
print(f"{'─'*48}")
if _records:
    preview = ", ".join(sorted(_records.keys())[:8])
    suffix  = f"  … +{len(_records)-8} more" if len(_records) > 8 else ""
    print(f"  Facilities : {preview}{suffix}")

# open the dashboard when history data is enough, or wait 5s
while True:
    with mqtt_data_lock:
        n_hist = len(facility_history)
        n_msg  = mqtt_message_count[0]
    if n_hist >= 100:
        break
    print(f"  History {n_hist}, not enough 100 data —> waiting 5s...")
    time.sleep(5)

print(f"\nHistory {n_hist}, messages {n_msg} —> enough data -> ready.")


────────────────────────────────────────────────
  Total MQTT messages     : 0
  Unique facilities       : 0
  History entries         : 0
  Unique timestamps       : 0
  Timestamps              : (none — run 3-2 first)
────────────────────────────────────────────────
  History 0, messages 0 —> waiting 5s...
  MQTT buffer: 22 msgs, 22 facilities
  History 38, messages 38 —> waiting 5s...
  History 90, messages 90 —> waiting 5s...
  History 137, messages 137 —> waiting 5s...
  History 184, messages 184 —> waiting 5s...
  History 232, messages 232 —> waiting 5s...
  History 279, messages 279 —> waiting 5s...
  History 324, messages 324 —> waiting 5s...
  History 381, messages 381 —> waiting 5s...
  History 427, messages 427 —> waiting 5s...
  History 475, messages 475 —> waiting 5s...

History 522, messages 522 —> enough data -> ready.


### 4-4 Map Dashboard

In [21]:
# clear all background running
_dashboard_poll_stop.set()
facility_records_by_code.clear()
map_markers_by_code.clear()
selected_facility_code[0] = None
current_map_popup[0]      = None

# display the dashboard
display(IPHTML(MapDash_CSS))
register_l1_toggle_buttons()
init_L2_map()
_dash_box.children = [L1_header, L2_map, L3_table, L4_date_label]

# update L3 table when dropdown changed
for btn, col in zip(L3_header_buttons, COLS):
    btn.on_click(lambda b, c=col: button_sort_column(c))

# update L2 map and L3 table when dropdown changed
for _dd in [L1_region_dropdown, L1_fuel_dropdown]:
    try: # clear existing dropdown
        _dd.unobserve(dropdown_update_map_and_table, names="value")
    except ValueError:
        pass
L1_region_dropdown.observe(dropdown_update_map_and_table, names="value")
L1_fuel_dropdown.observe(dropdown_update_map_and_table, names="value")

refresh_L3_header_buttons()
display(_dash_box)
time.sleep(0.2) # wait for rendering


# Load MQTT snapshot; create markers + populate map in batches 
with mqtt_data_lock:
    snapshot = dict(mqtt_facility_records)
facility_records_by_code.update(snapshot)
new_codes = [c for c in facility_records_by_code if c not in map_markers_by_code]

mode     = _toggle_state[0]
region_f = L1_region_dropdown.value
fuel_f   = L1_fuel_dropdown.value
accumulated      = []
first_batch_done = False

# draw circle markers in batches (20 at a time)
for i in range(0, max(len(new_codes), 1), 20):
    batch = new_codes[i:i + 20]
    if batch:
        newData_create_markers(batch)
    # update marker radius and color
    for code in batch:
        mk = map_markers_by_code.get(code)
        if mk is None:
            continue
        d = facility_records_by_code.get(code, {})
        mk.radius = marker_radius(d, mode)
        if (region_f == "All" or d.get("network_region") == region_f) and \
           (fuel_f   == "All" or d.get("primary_fuel")   == fuel_f):
            accumulated.append(mk)
    L2_marker_layer.layers = tuple(accumulated)
    
    if not first_batch_done:
        refresh_filter_dropdowns()
        render_facility_table()
        update_L4_date_label()
        first_batch_done = True # first batch done
    time.sleep(0.05)

# update L1 dropdowns and L3 table when first batch done
if not first_batch_done:
    refresh_filter_dropdowns()
    render_facility_table()
    update_L4_date_label()

# start getting new data
_poll_last_count[0] = 0 # for poll_refresh() check whether new data coming
_dashboard_poll_stop.clear()
threading.Timer(0.1, poll_refresh).start() # polling in new thread (can run in background)

print(f"Dashboard ready — {len(map_markers_by_code)} markers, MQTT buffer: {len(mqtt_facility_records)} facilities")

Dashboard ready — 326 markers, MQTT buffer: 326 facilities


## 5. Continuous Execution

### 5-1 Continuous Data Stream

In [22]:
# Shared state for section 6 dashboard
_round_data_lock     = threading.Lock()
_round_number        = [0]
_round_start_time    = [None]
_sleep_start_time    = [None]   # for concurrent round
_current_time_points = [0]      # number of 60 s buckets in latest chart render

# parameter initialisation
_cont_pub_stop = threading.Event()
_cont_pub_stop.set()


def continuous_publish():
    global _cont_pub_stop  # for other place (dashboard) use the same event
    _cont_pub_stop.set()   # reset the event
    _cont_pub_stop = threading.Event()  # for background running (multiple threads)
    stop = _cont_pub_stop
    n = 0
    while not stop.is_set():  # for safer stop
        n += 1
        with _round_data_lock:
            _round_number[0]     = n
            _round_start_time[0] = datetime.now()
            _sleep_start_time[0] = None
        label = f"{df_publish.shape[0]:,} records (full)" if n == 1 else "delta only"
        print(f"[Round {n}] publishing {label}\u2026")
        
        # use section 3 def, but in delta-only mode (only publish changed rows)
        _publish_loop(wait=0.1, delta=True)   
        print(f"[Round {n}] done. Sleeping 60 s\u2026")
        
        with _round_data_lock:
            _sleep_start_time[0] = datetime.now()
        stop.wait(60)


threading.Thread(target=continuous_publish, daemon=True).start()
print("Continuous execution started \u2014 delta-only updates per round, 60 s between rounds.")


[Round 1] publishing 657,206 records (full)…
Continuous execution started — delta-only updates per round, 60 s between rounds.


  Delta: 657,206 / 657,206 records changed


### 5-2 HTML and Def preparaion for Continuous Dashboard

In [23]:
cont_countdown_html = w.HTML(
    value=(
        "<div style='display:flex;flex-direction:column;align-items:center;"
        "justify-content:center;height:100%;padding:16px;box-sizing:border-box'>"
        "<span style='font-size:80px;font-weight:700;color:#3e4958;line-height:1'>—</span>"
        "<span style='font-size:12px;color:#888;margin-top:6px'>waiting to start</span>"
        "</div>"
    ),
    layout=w.Layout(flex="0 0 20%", min_width="140px", max_width="200px"),
)
cont_chart_html = w.HTML(
    value="<i style='color:#888'>Waiting for round to start…</i>",
    layout=w.Layout(flex="1 1 0%", min_width="0"),
)
cont_layout = w.HBox(
    [cont_countdown_html, cont_chart_html],
    layout=w.Layout(width="100%", align_items="stretch", min_height="260px"),
)

cont_countdown_html_beginning = \
                "<div style='display:flex;flex-direction:column;align-items:center;" \
                "justify-content:center;height:100%;padding:16px;box-sizing:border-box'>" \
                "<span style='font-size:14px;font-weight:600;color:#3e4958'>Waiting&#8230;</span>" \
                "</div>"

def cont_countdown_html_following(current_round, remaining, current_time_points):
    return (
        "<div style='display:flex;flex-direction:column;align-items:center;"
        "justify-content:center;height:100%;padding:16px;box-sizing:border-box'>"
        f"<span style='font-size:14px;font-weight:600;color:#3e4958;margin-bottom:6px'>Round {current_round}</span>"
        "<span style='font-size:12px;color:#888;margin-bottom:8px;text-align:center'>next update in (s)</span>"
        f"<span style='font-size:80px;font-weight:700;color:#3e4958;line-height:1'>{remaining}</span>"
        f"<span style='font-size:12px;color:#888;margin-top:10px'>{current_time_points} time point(s)</span>"
        "</div>"
    )


In [24]:
def cont_render_chart(chart_rn, pub_start_str):
    # get the data from mqtt
    with mqtt_data_lock:
        snapshot = list(facility_history.values())
    round_records = [r for r in snapshot if r.get("received_at", "") >= pub_start_str]
    if not round_records:
        cont_chart_html.value = "<i style='color:#888'>Publishing in progress…</i>"
        return

    pub_start_dt = datetime.fromisoformat(pub_start_str)
    bucket_fuel  = {}       

    # calculate the fuel count for each bucket
    for r in round_records:
        recv = r.get("received_at", "")
        fuel = (r.get("primary_fuel") or "unknown").lower()
        if not recv:
            continue
        elapsed     = (datetime.fromisoformat(recv) - pub_start_dt).total_seconds()
        bucket_dt   = pub_start_dt + timedelta(seconds=int(elapsed // 60) * 60)
        bucket_lbl  = bucket_dt.strftime("%H:%M:%S")
        bucket_fuel.setdefault(bucket_lbl, {})
        bucket_fuel[bucket_lbl][fuel] = bucket_fuel[bucket_lbl].get(fuel, 0) + 1

    all_buckets = sorted(bucket_fuel)
    _current_time_points[0] = len(all_buckets)
    all_fuels   = sorted(
        {f for d in bucket_fuel.values() for f in d},
        key=lambda f: sum(bucket_fuel[b].get(f, 0) for b in all_buckets), reverse=True,
    )
    xs = list(range(len(all_buckets)))
    if not xs:
        cont_chart_html.value = "<i style='color:#888'>Collecting data…</i>"
        return

    # calculate the stack for each fuel
    stacks = [[bucket_fuel[b].get(f, 0) for b in all_buckets] for f in all_fuels]
    colors = [FUEL_COLOR.get(f, DEFAULT_FUEL_COLOR) for f in all_fuels]

    # plot the chart
    fig = plt.Figure(figsize=(9, 4))
    fig.patch.set_facecolor("white")
    fig.subplots_adjust(left=0.08, right=0.97, top=0.88, bottom=0.16)
    ax = fig.add_subplot(111)

    if len(xs) == 1: # single bucket — bar chart
        bottoms = [0] * len(all_fuels)
        for j, (fuel, stack) in enumerate(zip(all_fuels, stacks)):
            ax.bar(xs, stack, bottom=[sum(s[0] for s in stacks[:j])],
                   color=colors[j], alpha=0.85,
                   label=fuel.replace("_", " ").title())
    else:
        ax.stackplot(xs, stacks,
                     labels=[f.replace("_", " ").title() for f in all_fuels],
                     colors=colors, alpha=0.85)
    ax.set_xticks(xs)
    ax.set_xticklabels(all_buckets, fontsize=8, rotation=20, ha="right")
    ax.tick_params(axis="y", labelsize=7)
    ax.set_title(f"Round {chart_rn} — Facilities Published per 60 s Window",
                 fontsize=10, loc="left", color="#333")
    ax.set_ylabel("Facility Count", fontsize=8)
    ax.set_xlim(-0.5, max(len(xs) - 0.5, 0.5))
    ax.grid(True, alpha=0.2, axis="y")
    hs, ls = ax.get_legend_handles_labels()
    if hs:
        ax.legend(hs[::-1], ls[::-1], fontsize=7, loc="upper left",
                  ncol=min(4, len(all_fuels)), framealpha=0.7)

    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=96, bbox_inches="tight")
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode()
    plt.close(fig)
    cont_chart_html.value = f"<img src='data:image/png;base64,{b64}' style='max-width:100%;'>"



### 5-3 Continuous Dashboard

In [25]:
def cont_update_thread(stop):
    chart_round   = 0
    current_round = 0  
    last_rn       = -1
    SEED_DELAY    = 2
    NORMAL_DELAY  = 60

    # delay for the chart
    def delay_for(cr):
        return SEED_DELAY if cr < 2 else NORMAL_DELAY
    last_chart_t = time.time() - delay_for(0)

    # keep updating the dashboard
    while not stop.is_set():
        with _round_data_lock:
            rn        = _round_number[0]
            pub_start = _round_start_time[0]

        now       = time.time()
        elapsed   = now - last_chart_t
        remaining = max(0, 59 - int(elapsed)) if chart_round >= 2 else "&#8212;"
        
        # countdown html: diff between first round and following rounds
        if rn == 0 or pub_start is None:
            cont_countdown_html.value = (cont_countdown_html_beginning)
        else:
            cont_countdown_html.value = (cont_countdown_html_following(current_round, remaining, _current_time_points[0]))

        # chart: update the chart
        if pub_start and (elapsed >= delay_for(chart_round) or rn != last_rn):
            if rn != last_rn:
                chart_round  = 0
                last_rn      = rn
                last_chart_t = time.time() - delay_for(0)
            else:
                chart_round += 1
            current_round = chart_round
            cont_render_chart(current_round, pub_start.isoformat(timespec="seconds"))
            last_chart_t = now

        stop.wait(1.0)

# foolproof: for cleaning the existing running threads
try:
    _cont_dash_stop.set()
except NameError:
    pass

# display the dashboard
_cont_dash_stop = threading.Event()
display(cont_layout)
threading.Thread(target=cont_update_thread, args=(_cont_dash_stop,), daemon=True).start()


## 6. Manual Cleanup When Needed

In [26]:
stop = False
#stop = True # manual clean for re-run section 5 & 6
if stop:
    try:
        sub_client.loop_stop()
        sub_client.disconnect()
    except Exception: pass

    _cont_pub_stop.set()
    _cont_dash_stop.set()

    try: _dashboard_poll_stop.set()   # section 5 — safe to skip if not defined
    except NameError: pass

    cont_chart_html.value    = "<i>Stopped.</i>"
    cont_countdown_html.value = (
        "<div class='cont-cd-wrap'><div class='cont-cd-num'>—</div></div>"
    )
    print("MQTT subscriber and dashboards stopped.")
else: 
    print("No Action Here")


No Action Here
